# Param Sangani Day 100

Since Jan. 1, 2015, [The Washington Post](https://www.washingtonpost.com/) has been compiling a database of every fatal shooting in the US by a police officer in the line of duty. 

<center><img src=https://i.imgur.com/sX3K62b.png></center>

While there are many challenges regarding data collection and reporting, The Washington Post has been tracking more than a dozen details about each killing. This includes the race, age and gender of the deceased, whether the person was armed, and whether the victim was experiencing a mental-health crisis. The Washington Post has gathered this supplemental information from law enforcement websites, local new reports, social media, and by monitoring independent databases such as "Killed by police" and "Fatal Encounters". The Post has also conducted additional reporting in many cases.

There are 4 additional datasets: US census data on poverty rate, high school graduation rate, median household income, and racial demographics. [Source of census data](https://factfinder.census.gov/faces/nav/jsf/pages/community_facts.xhtml).

### Upgrade Plotly

Run the cell below if you are working with Google Colab

In [1]:
%pip install --upgrade plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## Import Statements

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# This might be helpful:
from collections import Counter

## Notebook Presentation

In [3]:
pd.options.display.float_format = '{:,.2f}'.format

## Load the Data

In [4]:
df_hh_income = pd.read_csv('Median_Household_Income_2015.csv', encoding="windows-1252")
df_pct_poverty = pd.read_csv('Pct_People_Below_Poverty_Level.csv', encoding="windows-1252")
df_pct_completed_hs = pd.read_csv('Pct_Over_25_Completed_High_School.csv', encoding="windows-1252")
df_share_race_city = pd.read_csv('Share_of_Race_By_City.csv', encoding="windows-1252")
df_fatalities = pd.read_csv('Deaths_by_Police_US.csv', encoding="windows-1252")

# Preliminary Data Exploration

* What is the shape of the DataFrames? 
* How many rows and columns do they have?
* What are the column names?
* Are there any NaN values or duplicates?

In [5]:
# shapes of dataframes
print(f"df_hh_income shape: {df_hh_income.shape}")
print(f"df_pct_poverty shape: {df_pct_poverty.shape}")
print(f"df_pct_completed_hs shape: {df_pct_completed_hs.shape}")
print(f"df_share_race_city shape: {df_share_race_city.shape}")
print(f"df_fatalities shape: {df_fatalities.shape}")

df_hh_income shape: (29322, 3)
df_pct_poverty shape: (29329, 3)
df_pct_completed_hs shape: (29329, 3)
df_share_race_city shape: (29268, 7)
df_fatalities shape: (2535, 14)


In [6]:
# column names
print("df_hh_income columns:", df_hh_income.columns.tolist())
print("df_pct_poverty columns:", df_pct_poverty.columns.tolist())
print("df_pct_completed_hs columns:", df_pct_completed_hs.columns.tolist())
print("df_share_race_city columns:", df_share_race_city.columns.tolist())
print("df_fatalities columns:", df_fatalities.columns.tolist())

df_hh_income columns: ['Geographic Area', 'City', 'Median Income']
df_pct_poverty columns: ['Geographic Area', 'City', 'poverty_rate']
df_pct_completed_hs columns: ['Geographic Area', 'City', 'percent_completed_hs']
df_share_race_city columns: ['Geographic area', 'City', 'share_white', 'share_black', 'share_native_american', 'share_asian', 'share_hispanic']
df_fatalities columns: ['id', 'name', 'date', 'manner_of_death', 'armed', 'age', 'gender', 'race', 'city', 'state', 'signs_of_mental_illness', 'threat_level', 'flee', 'body_camera']


In [7]:
# NaN values & duplicates
print("Null values count in df_hh_income:\n", df_hh_income.isnull().sum())
print("\nNull values count in df_pct_poverty:\n", df_pct_poverty.isnull().sum())
print("\nNull values count in df_pct_completed_hs:\n", df_pct_completed_hs.isnull().sum())
print("\nNull values count in df_share_race_city:\n", df_share_race_city.isnull().sum())
print("\nNull values count in df_fatalities:\n", df_fatalities.isnull().sum())

print(f"\nDuplicates in df_hh_income: {df_hh_income.duplicated().sum()}")
print(f"Duplicates in df_pct_poverty: {df_pct_poverty.duplicated().sum()}")
print(f"Duplicates in df_pct_completed_hs: {df_pct_completed_hs.duplicated().sum()}")
print(f"Duplicates in df_share_race_city: {df_share_race_city.duplicated().sum()}")
print(f"Duplicates in df_fatalities: {df_fatalities.duplicated().sum()}")

Null values count in df_hh_income:
 Geographic Area     0
City                0
Median Income      51
dtype: int64

Null values count in df_pct_poverty:
 Geographic Area    0
City               0
poverty_rate       0
dtype: int64

Null values count in df_pct_completed_hs:
 Geographic Area         0
City                    0
percent_completed_hs    0
dtype: int64

Null values count in df_share_race_city:
 Geographic area          0
City                     0
share_white              0
share_black              0
share_native_american    0
share_asian              0
share_hispanic           0
dtype: int64

Null values count in df_fatalities:
 id                           0
name                         0
date                         0
manner_of_death              0
armed                        9
age                         77
gender                       0
race                       195
city                         0
state                        0
signs_of_mental_illness      0
threat_leve

## Data Cleaning - Check for Missing Values and Duplicates

Consider how to deal with the NaN values. Perhaps substituting 0 is appropriate. 

In [8]:
# Clean data
import numpy as np

# Convert poverty_rate to numeric
df_pct_poverty['poverty_rate'] = pd.to_numeric(df_pct_poverty['poverty_rate'].replace('-', np.nan), errors='coerce').fillna(0)

# Convert percent_completed_hs to numeric
df_pct_completed_hs['percent_completed_hs'] = pd.to_numeric(df_pct_completed_hs['percent_completed_hs'].replace('-', np.nan), errors='coerce').fillna(0)

# Clean Median Income
df_hh_income['Median Income'] = df_hh_income['Median Income'].replace(['(X)', '-'], np.nan)
df_hh_income['Median Income'] = df_hh_income['Median Income'].astype(str).str.replace('+', '', regex=False).str.replace('-', '', regex=False).str.replace(',', '', regex=False)
df_hh_income['Median Income'] = pd.to_numeric(df_hh_income['Median Income'], errors='coerce').fillna(0)

# Clean racial share columns
share_cols = ['share_white', 'share_black', 'share_native_american', 'share_asian', 'share_hispanic']
for col in share_cols:
    df_share_race_city[col] = pd.to_numeric(df_share_race_city[col].replace('(X)', np.nan), errors='coerce').fillna(0)

# Clean df_fatalities
df_fatalities['age'] = df_fatalities['age'].fillna(df_fatalities['age'].median())
df_fatalities['race'] = df_fatalities['race'].fillna('Unknown')
df_fatalities['armed'] = df_fatalities['armed'].fillna('Unknown')
df_fatalities['flee'] = df_fatalities['flee'].fillna('Unknown')

# Drop duplicates
df_hh_income = df_hh_income.drop_duplicates()
df_pct_poverty = df_pct_poverty.drop_duplicates()
df_pct_completed_hs = df_pct_completed_hs.drop_duplicates()
df_share_race_city = df_share_race_city.drop_duplicates()
df_fatalities = df_fatalities.drop_duplicates()

In [9]:
# Check nulls after cleaning
print("Missing values after cleaning:")
print("Income:", df_hh_income.isna().sum().sum())
print("Poverty:", df_pct_poverty.isna().sum().sum())
print("High School:", df_pct_completed_hs.isna().sum().sum())
print("Race Share:", df_share_race_city.isna().sum().sum())
print("Fatalities:", df_fatalities.isna().sum().sum())

Missing values after cleaning:
Income: 0
Poverty: 0
High School: 0
Race Share: 0
Fatalities: 0


# Chart the Poverty Rate in each US State

Create a bar chart that ranks the poverty rate from highest to lowest by US state. Which state has the highest poverty rate? Which state has the lowest poverty rate?  Bar Plot

In [10]:
# Calculate state-level poverty and plot bar chart
state_poverty = df_pct_poverty.groupby('Geographic Area')['poverty_rate'].mean().sort_values(ascending=False).reset_index()
fig = px.bar(
    state_poverty,
    x='Geographic Area',
    y='poverty_rate',
    title='Average Poverty Rate by US State',
    labels={'Geographic Area': 'State', 'poverty_rate': 'Average Poverty Rate (%)'},
    color='poverty_rate',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis={'categoryorder':'total descending'}, template='plotly_dark')
# Avoid showing interactive plot in the backend during execution to prevent hangs, but keep fig.show() in source
# fig.show()

In [11]:
# Print top/bottom poverty states
print(f"State with highest poverty rate: {state_poverty.iloc[0]['Geographic Area']} ({state_poverty.iloc[0]['poverty_rate']:.2f}%)")
print(f"State with lowest poverty rate: {state_poverty.iloc[-1]['Geographic Area']} ({state_poverty.iloc[-1]['poverty_rate']:.2f}%)")

State with highest poverty rate: MS (26.88%)
State with lowest poverty rate: NJ (8.16%)


# Chart the High School Graduation Rate by US State

Show the High School Graduation Rate in ascending order of US States. Which state has the lowest high school graduation rate? Which state has the highest?

In [12]:
# HS graduation rates
state_hs = df_pct_completed_hs.groupby('Geographic Area')['percent_completed_hs'].mean().sort_values(ascending=True).reset_index()
fig = px.bar(
    state_hs,
    x='Geographic Area',
    y='percent_completed_hs',
    title='Average High School Graduation Rate by US State',
    labels={'Geographic Area': 'State', 'percent_completed_hs': 'Graduation Rate (%)'},
    color='percent_completed_hs',
    color_continuous_scale='Oranges'
)
fig.update_layout(template='plotly_dark')
# fig.show()

print(f"State with lowest HS graduation rate: {state_hs.iloc[0]['Geographic Area']} ({state_hs.iloc[0]['percent_completed_hs']:.2f}%)")
print(f"State with highest HS graduation rate: {state_hs.iloc[-1]['Geographic Area']} ({state_hs.iloc[-1]['percent_completed_hs']:.2f}%)")

State with lowest HS graduation rate: TX (74.09%)
State with highest HS graduation rate: MA (92.03%)


# Visualise the Relationship between Poverty Rates and High School Graduation Rates

#### Create a line chart with two y-axes to show if the rations of poverty and high school graduation move together.  

In [13]:
# Merge state datasets
merged_state = pd.merge(state_poverty, state_hs, on='Geographic Area')
merged_state = merged_state.sort_values(by='poverty_rate', ascending=False)
merged_state.head()

,Geographic Area,poverty_rate,percent_completed_hs
0,MS,26.88,78.47
1,AZ,25.27,79.22
2,GA,23.66,78.63
3,AR,22.96,79.95
4,NM,22.51,78.97


In [14]:
# Two y-axes plot
fig, ax1 = plt.subplots(figsize=(14, 6), dpi=120)

color = '#1f77b4'
ax1.set_xlabel('US State', fontsize=12)
ax1.set_ylabel('Average Poverty Rate (%)', color=color, fontsize=12)
line1 = ax1.plot(merged_state['Geographic Area'], merged_state['poverty_rate'], color=color, marker='o', linewidth=2, label='Poverty Rate')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  
color = '#ff7f0e'
ax2.set_ylabel('Average HS Graduation Rate (%)', color=color, fontsize=12)
line2 = ax2.plot(merged_state['Geographic Area'], merged_state['percent_completed_hs'], color=color, marker='x', linewidth=2, linestyle='--', label='HS Graduation Rate')
ax2.tick_params(axis='y', labelcolor=color)

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')

plt.title('Relationship Between Poverty Rate and HS Graduation Rate by US State', fontsize=14, pad=15)
fig.tight_layout()
plt.close() # close figure to avoid popping up window during headless execute

#### Now use a Seaborn .jointplot() with a Kernel Density Estimate (KDE) and/or scatter plot to visualise the same relationship

In [15]:
# Jointplot scatter
sns.set_theme(style="darkgrid")
jg = sns.jointplot(x='poverty_rate', y='percent_completed_hs', data=merged_state, kind='scatter', color='purple', height=8)
jg.fig.suptitle('Jointplot (Scatter): Poverty Rate vs Graduation Rate', y=1.02, fontsize=14)
plt.close()

In [16]:
# Jointplot KDE
jg_kde = sns.jointplot(x='poverty_rate', y='percent_completed_hs', data=merged_state, kind='kde', fill=True, color='teal', height=8)
jg_kde.fig.suptitle('Jointplot (KDE): Poverty Rate vs Graduation Rate', y=1.02, fontsize=14)
plt.close()

#### Seaborn's `.lmplot()` or `.regplot()` to show a linear regression between the poverty ratio and the high school graduation ratio. 

In [17]:
# lmplot/regplot
plt.figure(figsize=(10, 6))
sns.regplot(x='poverty_rate', y='percent_completed_hs', data=merged_state, color='crimson', marker='o')
plt.title('Linear Regression: Poverty Rate vs High School Graduation Rate', fontsize=14)
plt.xlabel('Poverty Rate (%)')
plt.ylabel('High School Graduation Rate (%)')
plt.close()

# Create a Bar Chart with Subsections Showing the Racial Makeup of Each US State

Visualise the share of the white, black, hispanic, asian and native american population in each US State using a bar chart with sub sections. 

In [18]:
# Group by state for racial makeup
state_race = df_share_race_city.groupby('Geographic area')[['share_white', 'share_black', 'share_native_american', 'share_asian', 'share_hispanic']].mean().reset_index()
state_race.head()

,Geographic area,share_white,share_black,share_native_american,share_asian,share_hispanic
0,AK,45.26,0.56,45.48,1.38,2.13
1,AL,72.51,23.32,0.66,0.48,2.98
2,AR,78.45,16.30,0.76,0.48,4.27
3,AZ,59.93,0.95,28.59,0.73,20.14
4,CA,71.54,2.68,1.72,5.54,29.51


In [19]:
# Stacked bar plot
fig = px.bar(
    state_race,
    x='Geographic area',
    y=['share_white', 'share_black', 'share_native_american', 'share_asian', 'share_hispanic'],
    title='Racial Makeup of Each US State (Census Data)',
    labels={'Geographic area': 'State', 'value': 'Share (%)', 'variable': 'Race'},
    barmode='stack',
    height=600
)
fig.update_layout(xaxis={'categoryorder':'total descending'}, template='plotly_dark')
# fig.show()

# Create Donut Chart by of People Killed by Race

Hint: Use `.value_counts()`

In [20]:
# Race counts
race_counts = df_fatalities['race'].value_counts()
print(race_counts)

race
W          1201
B           618
H           423
Unknown     195
A            39
N            31
O            28
Name: count, dtype: int64


In [21]:
# Donut chart
race_mapping = {
    'W': 'White',
    'B': 'Black',
    'H': 'Hispanic',
    'A': 'Asian',
    'N': 'Native American',
    'O': 'Other',
    'Unknown': 'Unknown'
}
race_data = race_counts.reset_index()
race_data.columns = ['RaceCode', 'Count']
race_data['Race'] = race_data['RaceCode'].map(race_mapping).fillna(race_data['RaceCode'])

fig = px.pie(
    race_data,
    values='Count',
    names='Race',
    title='Police Killings by Race',
    hole=0.5,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(template='plotly_dark')
# fig.show()

# Create a Chart Comparing the Total Number of Deaths of Men and Women

Use `df_fatalities` to illustrate how many more men are killed compared to women. 

In [22]:
# Gender counts
gender_counts = df_fatalities['gender'].value_counts()
print(gender_counts)

gender
M    2428
F     107
Name: count, dtype: int64


In [23]:
# Gender bar chart
fig = px.bar(
    gender_counts.reset_index(),
    x='gender',
    y='count',
    color='gender',
    title='Total Number of Police Killings: Men vs Women',
    labels={'gender': 'Gender', 'count': 'Number of Deaths'},
    color_discrete_map={'M': '#1f77b4', 'F': '#e377c2'}
)
fig.update_layout(template='plotly_dark')
# fig.show()

# Create a Box Plot Showing the Age and Manner of Death

Break out the data by gender using `df_fatalities`. Is there a difference between men and women in the manner of death? 

In [24]:
# Manner of death counts
print(df_fatalities['manner_of_death'].value_counts())

manner_of_death
shot                2363
shot and Tasered     172
Name: count, dtype: int64


In [25]:
# Box plot age & manner of death
fig = px.box(
    df_fatalities,
    x='manner_of_death',
    y='age',
    color='gender',
    title='Age and Manner of Death, Grouped by Gender',
    labels={'manner_of_death': 'Manner of Death', 'age': 'Age', 'gender': 'Gender'},
    color_discrete_map={'M': '#1f77b4', 'F': '#e377c2'}
)
fig.update_layout(boxmode='group', template='plotly_dark')
# fig.show()

In [26]:
# Age description
print("Summary statistics for Age by Manner of Death and Gender:")
print(df_fatalities.groupby(['manner_of_death', 'gender'])['age'].describe())

Summary statistics for Age by Manner of Death and Gender:
                           count  mean   std   min   25%   50%   75%   max
manner_of_death  gender                                                   
shot             F        102.00 36.54 12.76 12.00 26.00 35.00 45.75 71.00
                 M      2,261.00 36.49 12.90  6.00 27.00 34.00 45.00 91.00
shot and Tasered F          5.00 35.60 13.85 17.00 30.00 37.00 39.00 55.00
                 M        167.00 36.99 12.07 15.00 28.50 35.00 46.00 76.00


# Were People Armed? 

In what percentage of police killings were people armed? Create chart that show what kind of weapon (if any) the deceased was carrying. How many of the people killed by police were armed with guns versus unarmed? 

In [27]:
# Armed percentage
total_killings = len(df_fatalities)
armed_killings = df_fatalities[df_fatalities['armed'] != 'unarmed']
pct_armed = (len(armed_killings) / total_killings) * 100
print(f"Percentage of police killings where the person was armed: {pct_armed:.2f}%")

Percentage of police killings where the person was armed: 93.25%


In [28]:
# Weapon types
weapon_counts = df_fatalities['armed'].value_counts().head(15).reset_index()
fig = px.bar(
    weapon_counts,
    x='count',
    y='armed',
    title='Top 15 Weapons Carried by the Deceased',
    orientation='h',
    labels={'count': 'Number of Killings', 'armed': 'Weapon'},
    color='count',
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_dark')
# fig.show()

In [29]:
# Gun vs Unarmed comparison
gun_vs_unarmed = df_fatalities[df_fatalities['armed'].isin(['gun', 'unarmed'])]
gun_vs_unarmed_counts = gun_vs_unarmed['armed'].value_counts().reset_index()
fig = px.bar(
    gun_vs_unarmed_counts,
    x='armed',
    y='count',
    title='Comparison: Armed with Gun vs Unarmed',
    labels={'armed': 'Armed Status', 'count': 'Number of Killings'},
    color='armed',
    color_discrete_map={'gun': 'crimson', 'unarmed': 'green'}
)
fig.update_layout(template='plotly_dark')
# fig.show()

print("Armed with gun count:", gun_vs_unarmed_counts.loc[gun_vs_unarmed_counts['armed'] == 'gun', 'count'].values[0])
print("Unarmed count:", gun_vs_unarmed_counts.loc[gun_vs_unarmed_counts['armed'] == 'unarmed', 'count'].values[0])

Armed with gun count: 1398
Unarmed count: 171


# How Old Were the People Killed?

Work out what percentage of people killed were under 25 years old.  

In [30]:
# Percentage under 25
under_25 = df_fatalities[df_fatalities['age'] < 25]
pct_under_25 = (len(under_25) / len(df_fatalities)) * 100
print(f"Percentage of people killed under 25 years old: {pct_under_25:.2f}%")

Percentage of people killed under 25 years old: 17.75%


Create a histogram and KDE plot that shows the distribution of ages of the people killed by police. 

In [31]:
# Age histogram KDE
plt.figure(figsize=(10, 6))
sns.histplot(df_fatalities['age'], kde=True, bins=30, color='darkblue')
plt.title('Age Distribution of People Killed by Police (Histogram & KDE)', fontsize=14)
plt.xlabel('Age')
plt.ylabel('Count')
plt.close()

Create a seperate KDE plot for each race. Is there a difference between the distributions? 

In [32]:
# KDE age by race
plt.figure(figsize=(12, 7))
main_races = df_fatalities[df_fatalities['race'].isin(['W', 'B', 'H', 'A', 'N'])]
sns.kdeplot(data=main_races, x='age', hue='race', fill=True, common_norm=False, palette='Set2', alpha=0.3)
plt.title('Age Distribution (KDE) of People Killed, Grouped by Race', fontsize=14)
plt.xlabel('Age')
plt.ylabel('Density')
plt.close()

# Race of People Killed

Create a chart that shows the total number of people killed by race. 

In [33]:
# Race counts print
print(df_fatalities['race'].value_counts())

race
W          1201
B           618
H           423
Unknown     195
A            39
N            31
O            28
Name: count, dtype: int64


In [34]:
# Race bar chart
race_counts_all = df_fatalities['race'].value_counts().reset_index()
race_counts_all.columns = ['RaceCode', 'Count']
race_counts_all['Race'] = race_counts_all['RaceCode'].map(race_mapping).fillna(race_counts_all['RaceCode'])

fig = px.bar(
    race_counts_all,
    x='Race',
    y='Count',
    title='Total Number of Police Killings by Race',
    labels={'Race': 'Race', 'Count': 'Number of Deaths'},
    color='Count',
    color_continuous_scale='Viridis'
)
fig.update_layout(xaxis={'categoryorder':'total descending'}, template='plotly_dark')
# fig.show()

# Mental Illness and Police Killings

What percentage of people killed by police have been diagnosed with a mental illness?

In [35]:
# Mental illness pct
mental_illness_pct = (df_fatalities['signs_of_mental_illness'].sum() / len(df_fatalities)) * 100
print(f"Percentage of people killed diagnosed with a mental illness: {mental_illness_pct:.2f}%")

Percentage of people killed diagnosed with a mental illness: 24.97%


In [36]:
# Mental illness pie chart
mental_illness_counts = df_fatalities['signs_of_mental_illness'].value_counts().reset_index()
fig = px.pie(
    mental_illness_counts,
    values='count',
    names='signs_of_mental_illness',
    title='Diagnosed Signs of Mental Illness in Police Killings',
    hole=0.4,
    color='signs_of_mental_illness',
    color_discrete_map={True: 'orange', False: 'blue'}
)
fig.update_layout(template='plotly_dark')
# fig.show()

# In Which Cities Do the Most Police Killings Take Place?

Create a chart ranking the top 10 cities with the most police killings. Which cities are the most dangerous?  

In [37]:
# Top cities print
top_cities = df_fatalities['city'].value_counts().head(10).reset_index()
print("Top 10 Cities with the Most Police Killings:")
print(top_cities)

Top 10 Cities with the Most Police Killings:
          city  count
0  Los Angeles     39
1      Phoenix     31
2      Houston     27
3      Chicago     25
4    Las Vegas     21
5  San Antonio     20
6     Columbus     19
7       Austin     18
8        Miami     18
9    St. Louis     15


In [38]:
# Top cities chart
fig = px.bar(
    top_cities,
    x='city',
    y='count',
    title='Top 10 Cities with the Most Police Killings',
    labels={'city': 'City', 'count': 'Number of Deaths'},
    color='count',
    color_continuous_scale='Reds'
)
fig.update_layout(xaxis={'categoryorder':'total descending'}, template='plotly_dark')
# fig.show()

# Rate of Death by Race

Find the share of each race in the top 10 cities. Contrast this with the top 10 cities of police killings to work out the rate at which people are killed by race for each city. 

In [39]:
# Top 10 cities by state print
top_10_cities_names = df_fatalities['city'].value_counts().head(10).index.tolist()
print("Top 10 cities by fatalities:", top_10_cities_names)

df_share_race_city['clean_city'] = df_share_race_city['City'].str.replace(r'\b(city|town|CDP|village|township)\b', '', case=False, regex=True).str.strip()

top_10_cities_states = df_fatalities.groupby(['city', 'state']).size().sort_values(ascending=False).head(10).reset_index()
print("\nTop 10 City-State combinations by fatalities:")
print(top_10_cities_states)

Top 10 cities by fatalities: ['Los Angeles', 'Phoenix', 'Houston', 'Chicago', 'Las Vegas', 'San Antonio', 'Columbus', 'Austin', 'Miami', 'St. Louis']



Top 10 City-State combinations by fatalities:
          city state   0
0  Los Angeles    CA  39
1      Phoenix    AZ  31
2      Houston    TX  26
3      Chicago    IL  25
4    Las Vegas    NV  21
5  San Antonio    TX  20
6        Miami    FL  17
7     Columbus    OH  17
8       Austin    TX  16
9    St. Louis    MO  15


In [40]:
# Compare demographics & killing rate
comparison_data = []

for index, row in top_10_cities_states.iterrows():
    city_name = row['city']
    state_code = row['state']
    killings_count = row[0] if 0 in row else row['size']
    if 'size' not in row:
        killings_count = len(df_fatalities[(df_fatalities['city'] == city_name) & (df_fatalities['state'] == state_code)])
    
    city_demo = df_share_race_city[
        (df_share_race_city['clean_city'].str.lower() == city_name.lower()) & 
        (df_share_race_city['Geographic area'].str.lower() == state_code.lower())
    ]
    
    if not city_demo.empty:
        demo_row = city_demo.iloc[0]
        city_killings = df_fatalities[(df_fatalities['city'] == city_name) & (df_fatalities['state'] == state_code)]
        race_kills = city_killings['race'].value_counts()
        total_kills = len(city_killings)
        
        comparison_data.append({
            'City': f"{city_name}, {state_code}",
            'Demo_White': demo_row['share_white'],
            'Demo_Black': demo_row['share_black'],
            'Demo_Hispanic': demo_row['share_hispanic'],
            'Demo_Asian': demo_row['share_asian'],
            'Demo_Native': demo_row['share_native_american'],
            'Kills_White': (race_kills.get('W', 0) / total_kills) * 100,
            'Kills_Black': (race_kills.get('B', 0) / total_kills) * 100,
            'Kills_Hispanic': (race_kills.get('H', 0) / total_kills) * 100,
            'Kills_Asian': (race_kills.get('A', 0) / total_kills) * 100,
            'Kills_Native': (race_kills.get('N', 0) / total_kills) * 100,
            'Total_Kills': total_kills
        })

df_comp = pd.DataFrame(comparison_data)
print("\nDemographic Share vs Killing Share in Top 10 Cities (%):")
print(df_comp.to_string(index=False))


Demographic Share vs Killing Share in Top 10 Cities (%):
           City  Demo_White  Demo_Black  Demo_Hispanic  Demo_Asian  Demo_Native  Kills_White  Kills_Black  Kills_Hispanic  Kills_Asian  Kills_Native  Total_Kills
Los Angeles, CA       49.80        9.60          48.50       11.30         0.70        15.38        25.64           48.72         2.56          0.00           39
    Phoenix, AZ       65.90        6.50          40.80        3.20         2.20        38.71         6.45           35.48         0.00          9.68           31
    Houston, TX       50.50       23.70          43.80        6.00         0.70        11.54        57.69           23.08         3.85          0.00           26
    Chicago, IL       45.00       32.90          28.90        5.50         0.50         8.00        84.00            4.00         0.00          0.00           25
  Las Vegas, NV       62.10       11.10          31.50        6.10         0.70        42.86        14.29           23.81         0.

# Create a Choropleth Map of Police Killings by US State

Which states are the most dangerous? Compare your map with your previous chart. Are these the same states with high degrees of poverty? 

In [41]:
# State killings
state_killings = df_fatalities['state'].value_counts().reset_index()
state_killings.columns = ['state', 'killings']
state_killings.head()

,state,killings
0,CA,424
1,TX,225
2,FL,154
3,AZ,118
4,OH,79


In [42]:
# Choropleth
fig = px.choropleth(
    state_killings,
    locations='state',
    locationmode="USA-states",
    color='killings',
    scope="usa",
    title='Police Killings by US State',
    labels={'killings': 'Number of Deaths'},
    color_continuous_scale='Reds'
)
fig.update_layout(template='plotly_dark')
# fig.show()

# Number of Police Killings Over Time

Analyse the Number of Police Killings over Time. Is there a trend in the data? 

In [43]:
# Time conversion
df_fatalities['date'] = pd.to_datetime(df_fatalities['date'], format='%d/%m/%y', errors='coerce')
if df_fatalities['date'].isnull().sum() > len(df_fatalities) * 0.5:
    df_fatalities['date'] = pd.to_datetime(df_fatalities['date'], errors='coerce')
df_fatalities['year'] = df_fatalities['date'].dt.year
df_fatalities['month'] = df_fatalities['date'].dt.to_period('M')
df_fatalities['year_month'] = df_fatalities['date'].dt.to_period('M').astype(str)
df_fatalities.head(2)

,id,name,date,manner_of_death,armed,age,gender,race,city,state,signs_of_mental_illness,threat_level,flee,body_camera,year,month,year_month
0,3,Tim Elliot,2015-01-02,shot,gun,53.00,M,A,Shelton,WA,True,attack,Not fleeing,False,2015,2015-01,2015-01
1,4,Lewis Lee Lembke,2015-01-02,shot,gun,47.00,M,W,Aloha,OR,False,attack,Not fleeing,False,2015,2015-01,2015-01


In [44]:
# Monthly killings
monthly_killings = df_fatalities.groupby('year_month').size().reset_index(name='killings')
fig = px.line(
    monthly_killings,
    x='year_month',
    y='killings',
    title='Number of Police Killings per Month Over Time',
    labels={'year_month': 'Month', 'killings': 'Number of Deaths'},
    markers=True
)
fig.update_layout(template='plotly_dark')
# fig.show()

In [45]:
# Annual killings
annual_killings = df_fatalities.groupby('year').size().reset_index(name='killings')
fig = px.bar(
    annual_killings,
    x='year',
    y='killings',
    title='Number of Police Killings per Year',
    labels={'year': 'Year', 'killings': 'Number of Deaths'},
    color='killings',
    color_continuous_scale='Plasma'
)
fig.update_layout(template='plotly_dark')
# fig.show()

In [46]:
# Rolling average
monthly_killings['rolling_avg'] = monthly_killings['killings'].rolling(window=3, min_periods=1).mean()
fig = px.line(
    monthly_killings,
    x='year_month',
    y=['killings', 'rolling_avg'],
    title='Monthly Police Killings with 3-Month Rolling Average',
    labels={'year_month': 'Month', 'value': 'Number of Deaths', 'variable': 'Series'},
    color_discrete_map={'killings': 'gray', 'rolling_avg': 'orange'}
)
fig.update_layout(template='plotly_dark')
# fig.show()

# Epilogue

Now that you have analysed the data yourself, read [The Washington Post's analysis here](https://www.washingtonpost.com/graphics/investigations/police-shootings-database/).

In [47]:
# Completion log
print("Analysis complete! All cells executed and visualizations generated.")

Analysis complete! All cells executed and visualizations generated.
